In [8]:
!pip install pyspark
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("L1_Apache_Spark").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def find_file(filename: str) -> Path:
    candidates = [Path.cwd() / filename, Path.cwd() / "data" / filename, Path("/content") / filename]
    for path in candidates:
        if path.exists(): return path
    raise FileNotFoundError(f"Файл {filename} не найден. Загрузите его в Colab.")

trips_raw = spark.read.option("header", True).csv(str(find_file("trip.csv")))
stations_raw = spark.read.option("header", True).csv(str(find_file("station.csv")))

trips_df = trips_raw.select(
    F.col("id").cast("int"),
    F.col("duration").cast("int"),
    F.coalesce(F.to_timestamp("start_date", "M/d/yyyy H:mm"), F.to_timestamp("start_date", "M/d/yyyy H:mm:ss")).alias("start_date"),
    "start_station_name", "start_station_id",
    F.coalesce(F.to_timestamp("end_date", "M/d/yyyy H:mm"), F.to_timestamp("end_date", "M/d/yyyy H:mm:ss")).alias("end_date"),
    "end_station_name", "end_station_id",
    F.col("bike_id").cast("int"),
    F.trim("subscription_type").alias("subscription_type"),
    F.trim("zip_code").alias("zip_code")
).where("id is not null and duration is not null and bike_id is not null").cache()

stations_df = stations_raw.select(
    F.col("id").cast("int"), "name",
    F.col("lat").cast("double"), F.col("long").cast("double"),
    F.col("dock_count").cast("int"), "city",
    F.coalesce(F.to_date("installation_date", "M/d/yyyy"), F.to_date("installation_date", "yyyy-MM-dd")).alias("installation_date")
).where("id is not null and lat is not null and long is not null").cache()

print(f"Загружено поездок: {trips_df.count()}, станций: {stations_df.count()}")

Загружено поездок: 669959, станций: 70


In [9]:
bike_time_stats = trips_df.groupBy("bike_id") \
    .agg(F.sum("duration").alias("ride_time_sec")) \
    .sort(F.col("ride_time_sec").desc(), F.col("bike_id").asc())

leader_bike = bike_time_stats.collect()[0]
bike_id_with_max_time = leader_bike["bike_id"]

print(f"ID велосипеда с наибольшим временем: {bike_id_with_max_time}")
print(f"Общее время: {leader_bike['ride_time_sec']} сек. ({leader_bike['ride_time_sec']/3600:.2f} ч.)")

ID велосипеда с наибольшим временем: 535
Общее время: 18611693 сек. (5169.91 ч.)


In [10]:
from math import radians, cos, sin, asin, sqrt

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * asin(sqrt(a))
    return R * c

haversine_udf = F.udf(haversine)

station_pairs = stations.alias("s1").crossJoin(stations.alias("s2")) \
    .filter("s1.id < s2.id")

max_dist = station_pairs.withColumn("dist",
    haversine_udf(F.col("s1.lat"), F.col("s1.long"), F.col("s2.lat"), F.col("s2.long")).cast("double")) \
    .select(F.max("dist").alias("max_distance_km"))

max_dist.show()

+-----------------+
|  max_distance_km|
+-----------------+
|69.92087595428183|
+-----------------+



In [11]:
EARTH_RADIUS_KM = 6371.0088

st_left = stations_df.selectExpr("id as id_l", "name as n_l", "city as c_l", "radians(lat) as lat_l", "radians(long) as lon_l")
st_right = stations_df.selectExpr("id as id_r", "name as n_r", "city as c_r", "radians(lat) as lat_r", "radians(long) as lon_r")

dist_expr = (2 * F.lit(EARTH_RADIUS_KM) *
    F.asin(F.sqrt(
        F.pow(F.sin((F.col("lat_r") - F.col("lat_l")) / 2), 2) +
        F.cos("lat_l") * F.cos("lat_r") * F.pow(F.sin((F.col("lon_r") - F.col("lon_l")) / 2), 2)
    )))

farthest_pair = st_left.crossJoin(st_right).filter("id_l < id_r") \
    .withColumn("dist", dist_expr).sort(F.col("dist").desc()).first()

print(f"Самое большое расстояние: {farthest_pair['dist']:.2f} км")
print(f"Между: {farthest_pair['n_l']} и {farthest_pair['n_r']}")

Самое большое расстояние: 69.92 км
Между: SJSU - San Salvador at 9th и Embarcadero at Sansome


In [12]:
bike_path = trips_df.filter(F.col("bike_id") == bike_id_with_max_time).sort("start_date")
path_data = bike_path.select("start_station_name", "end_station_name").collect()

full_route = [path_data[0]["start_station_name"]] + [i["end_station_name"] for i in path_data]
print(f"Количество поездок лидера: {len(path_data)}")
print("Маршрут:", " -> ".join(full_route[:10]), "...")

bike_count = trips_df.select("bike_id").distinct().count()
print(f"\nВсего велосипедов в системе: {bike_count}")

Количество поездок лидера: 1328
Маршрут: Post at Kearney -> San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend) -> Market at Sansome -> 2nd at South Park -> Davis at Jackson -> Civic Center BART (7th at Market) -> Post at Kearney -> Embarcadero at Sansome -> Washington at Kearney ...

Всего велосипедов в системе: 700


In [13]:
users_over_3h = trips_df.filter("zip_code is not null and zip_code != ''") \
    .groupBy("zip_code").agg(F.sum("duration").alias("sum_sec")) \
    .filter("sum_sec > 10800") \
    .withColumn("hours", F.round(F.col("sum_sec") / 3600, 2)) \
    .sort(F.desc("sum_sec"))

print("Пользователи (zip_code) с пробегом > 3 часов:")
users_over_3h.show(10)

zips = [row['zip_code'] for row in users_over_3h.select("zip_code").collect()]
print(f"Найдено пользователей: {len(zips)}")
print("Список первых 20 ZIP:", ", ".join(zips[:20]))

Пользователи (zip_code) с пробегом > 3 часов:
+--------+--------+--------+
|zip_code| sum_sec|   hours|
+--------+--------+--------+
|   94107|49757162|13821.43|
|     nil|45725550|12701.54|
|   94105|25596128| 7110.04|
|   94133|21637675| 6010.47|
|   94102|19128021| 5313.34|
|   94103|19127388| 5313.16|
|   95531|17270400| 4797.33|
|   94111|14244997| 3956.94|
|   95112|12742370| 3539.55|
|   94109|12057128|  3349.2|
+--------+--------+--------+
only showing top 10 rows
Найдено пользователей: 3660
Список первых 20 ZIP: 94107, nil, 94105, 94133, 94102, 94103, 95531, 94111, 95112, 94109, 94040, 94110, 94117, 94301, 94041, 94158, 94306, 94025, 94108, 94611
